# 00. 전처리 (코드정리 기준)

- Description 결측: StockCode별 최빈값
- 비정상/서비스 StockCode 제거 (DOT, M, POST, BANK CHARGES, C2, PADS, B, S, AMAZONFEE, m, gift)
- 취소 주문 및 매칭된 구매 제거
- Description 최빈값 통일 (merge), UnitPrice 0 보정, StockCode 5자리 숫자만 유지
- CustomerID·UnitPrice 결측 제거, 무의미 Description 제거
- 산출: `전처리_완료.csv`

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../데이터셋/Online_Retail.csv")
print(f"로드 행 수: {len(df):,}")

# Description 결측치 채우기 (StockCode별 최빈값)
df['Description'] = df.groupby('StockCode')['Description'].transform(
    lambda x: x.fillna(x.mode().iloc[0]) if not x.mode().empty else x
)

# 분석에서 제외할 비정상/서비스용 StockCode 정의 및 제거
remove_codes = ['DOT', 'M', 'POST', 'BANK CHARGES', 'C2', 'PADS', 'B', 'S', 'AMAZONFEE', 'm']
df = df[~df['StockCode'].isin(remove_codes)]
df = df[~df['StockCode'].astype(str).str.contains('gift', na=False)]

df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# 구매와 취소 분리
cancel = df[df['InvoiceNo'].str.startswith('C')].copy()
purchase = df[~df['InvoiceNo'].str.startswith('C')].copy()

cancel = cancel.reset_index().rename(columns={'index': 'cancel_idx'})
purchase = purchase.reset_index().rename(columns={'index': 'purchase_idx'})
cancel['abs_qty'] = cancel['Quantity'].abs()

# 고객ID, 상품코드, 수량 동일 매칭 (구매일 <= 취소일)
matched = cancel.merge(purchase, left_on=['CustomerID', 'StockCode', 'abs_qty'],
                       right_on=['CustomerID', 'StockCode', 'Quantity'], suffixes=('_c', '_p'))
matched = matched[matched['InvoiceDate_p'] <= matched['InvoiceDate_c']]

idx_to_remove_purchase = matched.sort_values('InvoiceDate_p').groupby('cancel_idx')['purchase_idx'].last()
remove_idx = set(cancel['cancel_idx']) | set(idx_to_remove_purchase)
df = df.drop(index=remove_idx, errors='ignore')

# Description 최빈값으로 통일
mode_desc = df.dropna(subset=['Description']).groupby('StockCode')['Description'].agg(
    lambda x: x.value_counts().idxmax()
)
df = df.drop(columns=['Description']).merge(mode_desc.rename('Description'), on='StockCode', how='inner')

# UnitPrice 0 → StockCode별 최빈 가격으로 대체
mode_price = df[df['UnitPrice'] > 0].groupby('StockCode')['UnitPrice'].agg(
    lambda x: x.value_counts().idxmax() if not x.empty else np.nan
)
df.loc[df['UnitPrice'] == 0, 'UnitPrice'] = df.loc[df['UnitPrice'] == 0, 'StockCode'].map(mode_price)

# StockCode 숫자 5자리로 시작하는 것만 유지
df = df[df['StockCode'].astype(str).str.match(r'^\d{5}')]

df = df.dropna(subset=['CustomerID', 'UnitPrice'])

remove_desc = ['High Resolution Image', 'Next Day Carriage']
df = df[~df['Description'].isin(remove_desc)]

print(f"최종 데이터 형태: {df.shape}")
print(df.isnull().sum())

df.to_csv('전처리_완료.csv', index=False, encoding='utf-8-sig')
print("\n✓ 전처리_완료.csv 저장 완료")

로드 행 수: 541,909
최종 데이터 형태: (393515, 8)
InvoiceNo      0
StockCode      0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Description    0
dtype: int64

✓ 전처리_완료.csv 저장 완료


## 핵심 요약

| 항목 | 결과 |
|------|------|
| **입력** | `Online_Retail.csv` (원본) |
| **처리** | Description 결측 보정, 비정상/서비스 StockCode 제거, 취소·매칭 구매 제거, UnitPrice 0 보정, CustomerID·UnitPrice 결측 제거 |
| **출력** | `전처리_완료.csv` |
| **최종 행 수** | 전처리 후 **393,515행** (회원 거래만), 8컬럼 |
| **산출물 위치** | 실행 시 작업 디렉터리(예: `분석 과정/`)에 저장됨. 이후 01·02·03에서 이 파일을 읽어 사용 |